In [164]:
import pandas as pd

In [165]:
# Load the dataset
df = pd.read_csv('Data/coffee_reviews_parsed.csv')

# Inspect the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9009 entries, 0 to 9008
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   URL                  9009 non-null   object 
 1   all_text             9009 non-null   object 
 2   Rating               9003 non-null   float64
 3   Roaster              9003 non-null   object 
 4   Coffee Name          9003 non-null   object 
 5   Roaster Location     8894 non-null   object 
 6   Coffee Origin        8436 non-null   object 
 7   Roast Level          8493 non-null   object 
 8   Agtron               8898 non-null   object 
 9   Est. Price           6859 non-null   object 
 10  Review Date          9008 non-null   object 
 11  Aroma                8828 non-null   float64
 12  Acidity              3709 non-null   float64
 13  Acidity/Structure    3809 non-null   float64
 14  Body                 8885 non-null   float64
 15  Flavor               8881 non-null   f

In [166]:
# Convert 'Review Date' to datetime and find the oldest and newest review dates
df['Review Date'] = pd.to_datetime(df['Review Date'].astype(str).str.strip(), format='%B %Y', errors='coerce')
oldest_review = df['Review Date'].min()
newest_review = df['Review Date'].max()

# Print the oldest and newest review dates, as well as the columns and the first few rows of the DataFrame
print(f'Oldest review date: {oldest_review.date()}')
print(f'Newest review date: {newest_review.date()}')

Oldest review date: 1997-02-01
Newest review date: 2026-03-01


In [167]:
# Check for duplicate URLs
print(f"Number of duplicate URLs: {df.duplicated(subset=['URL']).sum()}\n")

# Check for missing values
print("Missing values in each column:")
print(df.isna().sum())

Number of duplicate URLs: 0

Missing values in each column:
URL                       0
all_text                  0
Rating                    6
Roaster                   6
Coffee Name               6
Roaster Location        115
Coffee Origin           573
Roast Level             516
Agtron                  111
Est. Price             2150
Review Date               1
Aroma                   181
Acidity                5300
Acidity/Structure      5200
Body                    124
Flavor                  128
Aftertaste              982
With Milk              7790
Blind Assessment         12
Notes                     4
Who Should Drink It    4982
Bottom Line            4191
Combined_Acidity       1491
dtype: int64


# Normalize Blind Assessment

In [168]:
# Normalize the 'Blind Assessment' text data

print("Before normalization:")
print(df["Blind Assessment"].head())

df["Blind Assessment"] = (
    df["Blind Assessment"]
    .str.strip() # Remove leading and trailing whitespace
    .str.lower() # Convert to lowercase
    .str.replace(r"\s+", " ", regex=True)  # Replace multiple spaces with a single space
)

print("\nAfter normalization:")
print(df["Blind Assessment"].head())

Before normalization:
0    Evaluated as espresso. Smoothly round aroma: t...
1    Produced from an ESE pod on a FrancisFrancis! ...
2    Bittersweet but balanced; chocolaty. Dark choc...
3    Evaluated at proportions of 5 grams of instant...
4    In the aroma caramel and wet burned wood notes...
Name: Blind Assessment, dtype: object

After normalization:
0    evaluated as espresso. smoothly round aroma: t...
1    produced from an ese pod on a francisfrancis! ...
2    bittersweet but balanced; chocolaty. dark choc...
3    evaluated at proportions of 5 grams of instant...
4    in the aroma caramel and wet burned wood notes...
Name: Blind Assessment, dtype: object


# Remove duplicates, empty, and blind assessment with less than 100 characters.

In [169]:
before = len(df)

# Filter out rows where 'Blind Assessment' is empty or has fewer than 100 characters
s = df["Blind Assessment"].fillna("").astype(str).str.strip()
df = df[s.str.len() >= 100].copy()

print("Rows before:", before)
print("Rows after:", len(df))
print("Rows removed:", before - len(df))

Rows before: 9009
Rows after: 8899
Rows removed: 110


In [170]:
# Check for duplicates in 'Blind Assessment' and remove them
n_dupes = df.duplicated(subset=["Blind Assessment"]).sum()
print("Exact duplicate rows (excluding first occurrence):", n_dupes)

dupes_df = df[df.duplicated(subset=["Blind Assessment"], keep=False)] \
            .sort_values("Blind Assessment")

print("\nDuplicate rows:")
print(dupes_df[["Blind Assessment"]].head(30))

df = df.drop_duplicates(subset=["Blind Assessment"], keep="first")
print("\nDataFrame shape after removing duplicates:", df.shape)

Exact duplicate rows (excluding first occurrence): 10

Duplicate rows:
                                       Blind Assessment
5727  a high score but a relatively low wow quotient...
5395  a high score but a relatively low wow quotient...
8733  a light, bright, fragrantly smooth cup. when h...
8732  a light, bright, fragrantly smooth cup. when h...
5396  bright but well-matrixed acidity, full body, a...
5750  bright but well-matrixed acidity, full body, a...
1653  deeply rich, chocolaty and fruit-toned. raspbe...
1654  deeply rich, chocolaty and fruit-toned. raspbe...
8843  evaluated as a green coffee sample-roasted to ...
8844  evaluated as a green coffee sample-roasted to ...
924   reader peter lynagh nominated this coffee, rat...
923   reader peter lynagh nominated this coffee, rat...
5040  richly chocolaty, deeply sweet. dark chocolate...
2340  richly chocolaty, deeply sweet. dark chocolate...
6222  sweetly and richly fermenty: rye whisky, lilac...
6661  sweetly and richly fermenty

# Preprocess Coffee Origin

In [171]:
import re
import unicodedata
import numpy as np

def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9;,\s/-]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# explicit country names + place aliases
country_terms = {
    "Ethiopia": ["ethiopia"],
    "Colombia": ["colombia"],
    "Panama": ["panama"],
    "Kenya": ["kenya"],
    "Indonesia": ["indonesia"],
    "Guatemala": ["guatemala"],
    "Costa Rica": ["costa rica"],
    "El Salvador": ["el salvador"],
    "Rwanda": ["rwanda"],
    "Brazil": ["brazil"],
    "Honduras": ["honduras"],
    "Peru": ["peru"],
    "Taiwan": ["taiwan"],
    "Papua New Guinea": ["papua new guinea"],
    "Nicaragua": ["nicaragua"],
    "Burundi": ["burundi"],
    "Thailand": ["thailand"],
    "India": ["india"],
    "Mexico": ["mexico"],
    "Tanzania": ["tanzania"],
    "Yemen": ["yemen"],
    "Ecuador": ["ecuador"],
    "Bolivia": ["bolivia"],
    "Jamaica": ["jamaica"],
    "Uganda": ["uganda"],
    "Dominican Republic": ["dominican republic"],
    "Zambia": ["zambia"],
    "China": ["china"],
    "Vietnam": ["vietnam"],
    "Philippines": ["philippines"],
    "Malaysia": ["malaysia"],
    "Laos": ["laos"],
    "Zimbabwe": ["zimbabwe"],
    "Haiti": ["haiti"],
    "Puerto Rico": ["puerto rico"],
    "Nepal": ["nepal"],
    "Myanmar": ["myanmar"],
    "Cameroon": ["cameroon"],
    "Australia": ["australia"],
    "South Africa": ["south africa"],
    "Malawi": ["malawi"],
    "Venezuela": ["venezuela", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo"],
    "United Kingdom": ["united kingdom", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "hawaii", "hawai'i", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau"
    ],
}

# coffee region aliases
region_aliases = {
    "Ethiopia": [
        "yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo",
        "kochere", "hambela", "shakiso", "jimma", "limu", "oromia",
        "harrar", "kaffa", "bench maji", "bench-maji", "arbegona", "bensa"
    ],
    "Colombia": [
        "huila", "cauca", "narino", "nariño", "tolima", "quindio", "quindío",
        "caldas", "risaralda", "antioquia", "cundinamarca", "santander",
        "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
        "piendamó", "caicedonia", "san agustin", "san agustin", "armenia"
    ],
    "Panama": [
        "boquete", "chiriqui", "chiriquí", "volcan", "volcán", "jaramillo",
        "alto quiel", "paso ancho", "piedra candela", "canas verdes",
        "cañas verdes", "silla del pando", "renacimiento"
    ],
    "Kenya": [
        "nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a",
        "thika", "ruiru", "mathira", "meru", "nakuru", "gichugu", "karatina"
    ],
    "Indonesia": [
        "sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang",
        "toraja", "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"
    ],
    "Guatemala": [
        "huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "coban",
        "atitlan", "atitlán", "chimaltenango", "quiche", "quiché", "solola",
        "sololá", "palencia", "sacatepequez", "sacatepéquez", "san marcos",
        "hoja blanca", "cuilco"
    ],
    "Costa Rica": [
        "tarrazu", "tarrazu", "central valley", "west valley", "tres rios",
        "tres ríos", "naranjo", "dota", "poas", "poás", "alajuela",
        "brunca", "turrialba", "coto brus", "chirripo", "chirripó"
    ],
    "El Salvador": [
        "ahuachapan", "ahuachapán", "chalatenango", "apaneca", "ilamatepec",
        "santa ana", "el boqueron", "el boquerón", "quetzaltepec", "juayua", "juayúa", "ataco"
    ],
    "Rwanda": [
        "gakenke", "nyamasheke", "karongi", "huye", "nyamagabe",
        "rulindo", "gikongoro", "lake kivu"
    ],
    "Brazil": [
        "cerrado", "mogiana", "minas gerais", "mantiqueira",
        "sul de minas", "chapada diamantina", "carmo de minas"
    ],
    "Honduras": [
        "marcala", "copan", "copán", "ocotepeque", "comayagua", "intibuca",
        "intibucá", "santa barbara", "santa bárbara", "el paraiso", "el paraíso", "capucas"
    ],
    "Peru": [
        "cajamarca", "jaen", "jaén", "san ignacio", "chanchamayo",
        "cusco", "villa rica", "oxapampa", "junin", "junín", "puno"
    ],
    "Taiwan": [
        "alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"
    ],
    "Papua New Guinea": [
        "wahgi valley", "western highlands", "eastern highlands",
        "jiwaka", "kainantu", "okapa"
    ],
    "Nicaragua": [
        "jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"
    ],
    "Burundi": [
        "kayanza", "ngozi", "muramvya", "muyinga", "bururi"
    ],
    "Thailand": [
        "chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"
    ],
    "India": [
        "coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"
    ],
    "Mexico": [
        "chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"
    ],
    "Tanzania": [
        "mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"
    ],
    "Yemen": [
        "haraaz", "haraz", "sanaa", "sanaa", "bani matar", "hayma"
    ],
    "Ecuador": [
        "loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"
    ],
    "Bolivia": [
        "caranavi", "yungas", "la paz"
    ],
    "Jamaica": [
        "blue mountain", "blue mountains"
    ],
    "Uganda": [
        "rwenzori", "bugisu", "sipi falls"
    ],
    "Vietnam": [
        "lam dong", "quang tri", "quảng trị", "dalat", "cau dat"
    ],
    "Philippines": [
        "benguet", "bukidnon", "davao"
    ],
    "China": [
        "yunnan", "baoshan", "puer", "pu'er", "lincong"
    ],
    "Puerto Rico": [
        "utuado", "yauco", "adjuntas"
    ],
    "Saint Helena": [
        "saint helena", "st helena", "sandy bay valley"
    ],
    "Venezuela": [
        "mocoties valley", "merida state"
    ],
    "Malawi": [
        "malawi"
    ],
}

# region-only values -> not a country, so keep as missing
region_only_terms = [
    "central america",
    "south america",
    "latin america",
    "east africa",
    "central africa",
    "west africa",
    "africa",
    "asia",
    "the americas",
    "americas",
    "central and south america",
    "south and central america",
    "east and central africa",
    "various africa growing regions"
]

# unknown / non-specific values
noise_terms = [
    "not disclosed",
    "not discolosed",
    "not specified",
    "undisclosed",
    "blend",
    "espresso blend",
    "various"
]

def extract_countries(x):
    t = normalize_text(x)

    if t == "":
        return np.nan

    # unknown / non-specific
    if any(term in t for term in noise_terms):
        return np.nan

    # region-only labels
    # if all content is just broad regions, return NaN
    temp = t.replace(".", "").strip()
    region_parts = [p.strip() for p in re.split(r"[;,]", temp) if p.strip()]
    if region_parts and all(
        any(r in part for r in region_only_terms) or part == "undisclosed"
        for part in region_parts
    ):
        return np.nan

    found = []

    # explicit country/place aliases
    for country, terms in country_terms.items():
        if any(re.search(rf"\b{re.escape(term)}\b", t) for term in terms):
            found.append(country)

    # region aliases
    for country, terms in region_aliases.items():
        if country not in found and any(term in t for term in terms):
            found.append(country)

    if not found:
        return np.nan

    return "; ".join(found)

df["Country_all"] = df["Coffee Origin"].apply(extract_countries)
df["Country"] = df["Country_all"].str.split("; ").str[0]

In [172]:
# Drop rows where either 'Coffee Origin' or 'Country' is missing (NaN) or empty after stripping whitespace
df = df.dropna(subset=["Coffee Origin", "Country"])

df = df[
    ~df[["Coffee Origin", "Country"]]
    .astype(str)
    .apply(lambda col: col.str.strip().eq(""))
    .any(axis=1)
].reset_index(drop=True)

In [173]:
# Check if both columns have the same number
print("Same amount of values in both columns:", df["Country"].value_counts().sum() == df.shape[0])

Same amount of values in both columns: True


# Save to csv

In [174]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('Data/final_coffee_reviews.csv', index=False)

df_test = pd.read_csv('Data/final_coffee_reviews.csv')
print(df_test.shape)

(7585, 25)
